In [1]:
# Librería para manejar datos en tablas
import pandas as pd

# Librería para expresiones regulares
import re

# Librería para similitud de cadenas
from difflib import get_close_matches

# Para trabajar con valores nulos
import numpy as np

In [ ]:

archivo = "ContextoSocial(educaciónFinanciera) - Hoja 1.csv"

# Cargar el archivo
df = pd.read_csv(archivo)

# Ver las primeras filas
df.head()

,1. Número de cuenta:,edad,dependenciaEconomica,ingresoMensual,gastoEntretenimiento
0,420093200,24,Independiente,"$20,000","$2,000"
1,422018647,24,dependiente,8000,2000
2,317127133,25,Independiente,6000,2000
3,422041654,29,Independiente,30000,4000
4,317237896,24,Dependiente,al rededor de 3mil al mes,al rededor de 2mil


In [3]:
# Ver estructura general
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   1. Número de cuenta:  19 non-null     int64
 1   edad                  19 non-null     int64
 2   dependenciaEconomica  19 non-null     str  
 3   ingresoMensual        18 non-null     str  
 4   gastoEntretenimiento  19 non-null     str  
dtypes: int64(2), str(3)
memory usage: 892.0 bytes


In [4]:
# Ver nombres de columnas
df.columns

Index(['1. Número de cuenta:', 'edad', 'dependenciaEconomica',
       'ingresoMensual', 'gastoEntretenimiento'],
      dtype='str')

In [5]:
# Quitar espacios al inicio y final
df.columns = df.columns.str.strip()

# Convertir a minúsculas
df.columns = df.columns.str.lower()

df.columns

Index(['1. número de cuenta:', 'edad', 'dependenciaeconomica',
       'ingresomensual', 'gastoentretenimiento'],
      dtype='str')

In [6]:
def limpiar_texto(texto):
    if pd.isna(texto):
        return np.nan
    
    # Convertir a minúsculas
    texto = texto.lower()
    
    # Quitar espacios extra
    texto = texto.strip()
    
    # Quitar caracteres especiales (solo dejar letras y números)
    texto = re.sub(r'[^a-zA-Z0-9áéíóúñ\s]', '', texto)
    
    return texto

In [7]:
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].apply(limpiar_texto)

df.head()

C:\Users\melis\AppData\Local\Temp\ipykernel_17232\739613557.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


,1. número de cuenta:,edad,dependenciaeconomica,ingresomensual,gastoentretenimiento
0,420093200,24,independiente,20000,2000
1,422018647,24,dependiente,8000,2000
2,317127133,25,independiente,6000,2000
3,422041654,29,independiente,30000,4000
4,317237896,24,dependiente,al rededor de 3mil al mes,al rededor de 2mil


In [ ]:
df.isnull().sum()
# Manejo de valores nulos: Ver cuántos hay 

1. número de cuenta:    0
edad                    0
dependenciaeconomica    0
ingresomensual          1
gastoentretenimiento    0
dtype: int64

In [9]:
# 1) ¿Cuántas filas/columnas hay de verdad?
df.shape, len(df)


((19, 5), 19)

In [12]:
import re

def limpiar_dependencia(valor):
    if pd.isna(valor):
        return None
    
    valor = str(valor).lower().strip()
    
    # Si dice "si", convertir a dependiente
    if valor == "si":
        return "dependiente"
    
    # Buscar la palabra independiente
    if "independiente" in valor:
        return "independiente"
    
    # Buscar la palabra dependiente
    if "dependiente" in valor:
        return "dependiente"
    
    return valor

df["dependenciaeconomica"] = df["dependenciaeconomica"].apply(limpiar_dependencia)

df["dependenciaeconomica"].unique()

<StringArray>
['independiente', 'dependiente']
Length: 2, dtype: str

In [15]:
import pandas as pd
import numpy as np
from difflib import get_close_matches

ID_COL = "1. Número de cuenta:"
COL = "dependenciaeconomica"

assert "df" in globals(), "df no existe: ejecuta la celda donde cargas el CSV."

# Diagnóstico de columnas
print("Columnas:", list(df.columns))

# Parche típico: espacios invisibles en nombres de columnas
df.columns = df.columns.astype(str).str.strip()

if COL not in df.columns:
    print(f"❌ No encuentro la columna exacta: {COL!r}")
    print("Sugerencias parecidas:", get_close_matches(COL, list(df.columns), n=8, cutoff=0.2))
    raise KeyError(f"Falta {COL!r}. Ajusta COL al nombre real o renombra la columna.")
else:
    print(f"✅ Columna encontrada: {COL}")


Columnas: ['1. número de cuenta:', 'edad', 'dependenciaeconomica', 'ingresomensual', 'gastoentretenimiento']
✅ Columna encontrada: dependenciaeconomica


In [16]:
display(df[COL].value_counts(dropna=False))
print("Únicos (hasta 30):", pd.Series(df[COL].unique()).head(30).tolist())


dependenciaeconomica
dependiente      12
independiente     7
Name: count, dtype: int64

Únicos (hasta 30): ['independiente', 'dependiente']


In [17]:
df["ingresomensual"].unique()

<StringArray>
[                                  '20000',
                                    '8000',
                                    '6000',
                                   '30000',
               'al rededor de 3mil al mes',
                                   '12000',
                                        '',
                                       '5',
                                   '10000',
 'pues me va bien pero me podria ir mejor',
                                    '9000',
                                   '15000',
                                     '10k',
                                       nan,
                                '13 aprox',
                                    '1000',
                                    '5000',
                      'mo propio nada unu']
Length: 18, dtype: str

In [18]:
def limpiar_ingreso(valor):
    
    # Si es nulo, regresamos NaN
    if pd.isna(valor):
        return np.nan
    
    # Convertimos todo a minúsculas y quitamos espacios
    valor = str(valor).lower().strip()
    
    # Si contiene la palabra "nada" → 0
    if "nada" in valor:
        return 0
    
    # Quitamos comas
    valor = valor.replace(",", "")
    
    # Caso 1: valores con "k" (ejemplo 10k)
    if "k" in valor:
        numero = re.findall(r"\d+", valor)
        if numero:
            return int(numero[0]) * 1000
        else:
            return np.nan
    
    # Caso 2: valores con "mil" (ejemplo 3mil)
    if "mil" in valor:
        numero = re.findall(r"\d+", valor)
        if numero:
            return int(numero[0]) * 1000
        else:
            return np.nan
    
    # Extraemos solo números
    numero = re.findall(r"\d+", valor)
    
    # Si no hay números → NaN
    if not numero:
        return np.nan
    
    numero = int(numero[0])
    
    # Si el número es menor que 100 → lo interpretamos como miles
    if numero < 100:
        return numero * 1000
    
    return numero

In [19]:
df["ingresomensual_limpio"] = df["ingresomensual"].apply(limpiar_ingreso)

In [20]:
df[["ingresomensual", "ingresomensual_limpio"]]

,ingresomensual,ingresomensual_limpio
0,20000,20000.0
1,8000,8000.0
2,6000,6000.0
3,30000,30000.0
4,al rededor de 3mil al mes,3000.0
5,12000,12000.0
6,,NaN
7,5,5000.0
8,10000,10000.0
9,pues me va bien pero me podria ir mejor,NaN


In [21]:
# Diccionario para convertir palabras a número
numeros_palabras = {
    "cero": 0,
    "cien": 100,
    "ciento": 100,
    "doscientos": 200,
    "trescientos": 300,
    "cuatrocientos": 400,
    "quinientos": 500,
    "seiscientos": 600,
    "setecientos": 700,
    "ochocientos": 800,
    "novecientos": 900,
    "mil": 1000
}

def limpiar_gasto(valor):
    
    # Si está vacío → 0
    if pd.isna(valor):
        return 0
    
    # Convertimos todo a texto, minúsculas y quitamos espacios
    valor = str(valor).lower().strip()
    
    # f) Si dice "nada" → 0
    if "nada" in valor:
        return 0
    
    # e) Si es palabra completa (ej: "trescientos")
    if valor in numeros_palabras:
        return numeros_palabras[valor]
    
    # c) Caso "10k"
    if re.search(r"\d+\s*k", valor):
        numero = re.findall(r"\d+", valor)[0]
        return int(numero) * 1000
    
    # d) Caso "3mil"
    if re.search(r"\d+\s*mil", valor):
        numero = re.findall(r"\d+", valor)[0]
        return int(numero) * 1000
    
    # a) Quitar comas
    valor = valor.replace(",", "")
    
    # b y g) Quitar cualquier cosa que no sea número
    valor = re.sub(r"[^0-9]", "", valor)
    
    # Si queda vacío → 0
    if valor == "":
        return 0
    
    return int(valor)

In [22]:
df["gastoentretenimiento_limpio"] = df["gastoentretenimiento"].apply(limpiar_gasto)

In [23]:
df[["gastoentretenimiento", "gastoentretenimiento_limpio"]]

,gastoentretenimiento,gastoentretenimiento_limpio
0,2000,2000
1,2000,2000
2,2000,2000
3,4000,4000
4,al rededor de 2mil,2000
5,400,400
6,600,600
7,900,900
8,700,700
9,en promedio 2000,2000
